# cycleform — Phase 1 pilot

Demonstrates the one thing Phase 1 had to prove: the **same metric code** 
runs on a real OSM city and on a grown network. All logic lives in 
`src/cycleform/`; this notebook just calls it.

Kernel: **neatnetenv**.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import logging; logging.basicConfig(level=logging.INFO, format='%(message)s')

from cycleform.ingest import context_from_osm
from cycleform.metrics import REGISTRY, results_to_frame
from cycleform.synthetic import fake_grown_context

print(len(REGISTRY), 'metrics registered:', REGISTRY.names)

## 1. A real city, straight from OSM

`context_from_osm` geocodes the boundary, fetches the drive + cycle 
networks, runs neatnet on the roads, tags LTS, and returns a `PlaceContext`. 
Change the place name to try another city (small ones are quick).

In [ ]:
ctx = context_from_osm('City of Chester, United Kingdom', place_id='Chester')
print('source :', ctx.source)
print('layers :', sorted(ctx.available))
print('area   : %.1f km2 (%s)' % (ctx.built_up_area_km2, ctx.meta['area_note']))
print('road   : %d edges, %d nodes' % (ctx.road.n_edges, ctx.road.n_nodes))
print('bike   : %d edges, %d nodes' % (ctx.bike.n_edges, ctx.bike.n_nodes))

In [ ]:
real = REGISTRY.run(ctx)
results_to_frame(real)

## 2. A grown network (synthetic stand-in)

`fake_grown_context` builds a `PlaceContext` with the *same schema* the 
Chapter-5 growth model will produce. The registry call below is identical 
— no branching on where the network came from. That is the §2 invariant.

In [ ]:
grown = fake_grown_context()
results_to_frame(REGISTRY.run(grown))

## 3. Plot the road network (quick visual check)

Not journal-grade — just a sanity check that neatnet produced a sensible 
simplified network. Publication figures come later.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 7))
ctx.road.edges.plot(ax=ax, linewidth=0.5, color='0.4')
ctx.bike.edges.plot(ax=ax, linewidth=1.2, color='crimson')
ax.set_title(f'{ctx.place_id}: road (grey) + cycle infrastructure (red)')
ax.set_axis_off(); plt.show()

## Next

- **Phase 2**: harmonise the cycling-rate outcome table (OECD FUA / Eurostat / UK).
- **Phase 3**: full metric suite + relational metrics, GHSL built-up area, run at scale.

See `CLAUDE.md §9` for the phase gates.